In [1]:
from z3 import *
from collections import Counter

Token = Datatype('Token')
Token.declare('num', ('int', IntSort())) # 
Token.declare('plus')
Token.declare('div')
Token.declare('mul')
Token = Token.create()

Tree = Datatype('Tree')
Tree.declare('nil')
Tree.declare('node', ('token', Token), ('left', Tree), ('right', Tree))
Tree = Tree.create()

def FiniteArray(sort):
    FiniteArr = Datatype('FiniteArray_%s' % sort.name())
    FiniteArr.declare('make', ('arr', ArraySort(IntSort(),sort)), ('size', IntSort()))
    FiniteArr = FiniteArr.create()
    return FiniteArr

def DeclareList(sort):
    List = Datatype('List_%s' % sort.name())
    List.declare('cons', ('car', sort), ('cdr', List))
    List.declare('nil')
    return List.create()

IntList = DeclareList(IntSort())

FiniteIntArraySort = FiniteArray(IntSort())

def Abs(x):
    return If(x >= 0,x,-x)

appendo = RecFunction('appendo', IntList, IntList, IntList)
l = FreshConst(IntList)
s = FreshConst(IntList)
RecAddDefinition( appendo
                , [l, s]
                , If(IntList.nil == l ,
                     s,
                     IntList.cons(IntList.car(l), appendo(IntList.cdr(l), s))
                     )
                )

example1 = Tree.node(Token.plus,#valid
                     Tree.node(Token.num(1),Tree.nil,Tree.nil),
                     Tree.node(Token.num(2),Tree.nil,Tree.nil))
example2 = Tree.node(Token.div,#non valid
           Tree.node(Token.num(7), Tree.nil, Tree.nil),
            Tree.node(Token.plus,
                Tree.node(Token.num(-7), Tree.nil, Tree.nil),
                Tree.node(Token.num(7), Tree.nil, Tree.nil)))

example3 =  Tree.node(Token.div,#valid
                Tree.node(Token.num(2), Tree.nil, Tree.nil),
                Tree.node(Token.div,
                    Tree.node(Token.num(2), Tree.nil, Tree.nil),
                    Tree.node(Token.plus,
                        Tree.node(Token.num(2), Tree.nil, Tree.nil),
                        Tree.node(Token.num(2), Tree.nil, Tree.nil))))

t1 = FreshConst(Tree)
compute = RecFunction("compute", Tree, RealSort()) #precondition is_valid(t1)
RecAddDefinition(compute,[t1],If(Tree.token(t1)==Token.plus,    
                                 compute(Tree.left(t1))+compute(Tree.right(t1)),
                              If(Tree.token(t1)==Token.div,     
                                 compute(Tree.left(t1))/compute(Tree.right(t1)),
                              If(Tree.token(t1)==Token.mul,     
                                 compute(Tree.left(t1))*compute(Tree.right(t1)),                                
                              ToReal(Token.int(Tree.token(t1)))
))))
s = Solver()
s.add(compute(example3)==4)# compute(example3)==x will satisfy for every x if we will use IntSort instead of RealSort because of div
if s.check() != sat:
    print("wrong result")


# t1 = FreshConst(Tree)
# is_valid = RecFunction("is_valid", Tree, BoolSort())
# RecAddDefinition(is_valid,[t1],
#                 And(t1!=Tree.nil,
#                   If(Or( Tree.token(t1)==Token.plus,  Tree.token(t1)==Token.div,  Tree.token(t1)==Token.mul) ,         
#                         And(Tree.left(t1)!=Tree.nil,Tree.right(t1)!=Tree.nil,is_valid(Tree.left(t1)),is_valid(Tree.right(t1)),Implies(Tree.token(t1)==Token.div,compute(Tree.right(t1))!=0)),
#                         And(Tree.left(t1)==Tree.nil,Tree.right(t1)==Tree.nil )
# )))

t1 = FreshConst(Tree)
is_valid = RecFunction("is_valid", Tree, BoolSort())
RecAddDefinition(is_valid,[t1],
                If(t1==Tree.nil,
                   False,
                If(Or( Tree.token(t1)==Token.plus,  Tree.token(t1)==Token.div,  Tree.token(t1)==Token.mul) ,         
                    And(is_valid(Tree.left(t1)),is_valid(Tree.right(t1)),Implies(Tree.token(t1)==Token.div,compute(Tree.right(t1))!=0)),
                And(Tree.left(t1)==Tree.nil,Tree.right(t1)==Tree.nil )
)))

s = Solver()
s.add(is_valid(example1))
if s.check() != sat:
    print("non valid tree")

tt = FreshConst(Tree)
ti = FreshConst(IntSort())
is_all_same = RecFunction("is_all_same", Tree, IntSort(),IntSort())
RecAddDefinition(is_all_same,[tt,ti],
                If(tt==Tree.nil,                                                                
                   0,
                If(Or(Tree.token(tt)==Token.num(ti),Tree.token(tt)==Token.num(-ti)),#TODO support concatination too
                    1,
                If(Or(is_all_same(Tree.left(tt),ti)==-1,is_all_same(Tree.right(tt),ti)==-1),    
                   -1,
                If(Or(Tree.token(tt)==Token.plus,Tree.token(tt)==Token.div,Tree.token(tt)==Token.mul),
                    is_all_same(Tree.left(tt),ti)+is_all_same(Tree.right(tt),ti),
                -1
)))))

s = Solver()
s.add(is_all_same(example1,2)==-1)
if s.check() != sat:
    print("example1 contain different number")

t = FreshConst(Tree)
get_numbers = RecFunction("get_numbers", Tree, IntList)
RecAddDefinition(get_numbers,[t],
                If(t==Tree.nil,                                                                
                   IntList.nil,
                If(Or(Tree.token(t)==Token.plus,Tree.token(t)==Token.div,Tree.token(t)==Token.mul),
                    appendo(get_numbers(Tree.left(t)),get_numbers(Tree.right(t))),
                IntList.cons(Abs(Token.int(Tree.token(t))), appendo(get_numbers(Tree.left(t)),get_numbers(Tree.right(t)))),
)))

l=FreshConst(IntList)
n=FreshConst(IntSort())
get_count = RecFunction("get_count", IntList, IntSort(),IntSort())
RecAddDefinition(get_count,[l,n],
                    If(IntList.nil == l,
                        0,
                    (IntList.car(l)==n)+get_count(IntList.cdr(l),n)
))

l=FreshConst(IntList)
length = RecFunction("length", IntList, IntSort())
RecAddDefinition(length,[l],
                    If(IntList.nil == l,
                        0,
                    1+length(IntList.cdr(l))
))

s = Solver()
s.add(get_numbers(example1)==IntList.cons(1, IntList.cons(2, IntList.nil)))
if s.check() != sat:
    print("example1 contain different number")

def Equal(a,b):
    if isinstance(a, DatatypeRef):
        if a.sort().name()[:5]=="List_" and isinstance(b,list):
            counts = Counter(b)
            conds=[]
            for num,count in counts.items():
                conds.append(get_count(a,num)==count)
            conds.append(length(a)==len(b))
            return And(conds)

def Contains(a,b):
    if isinstance(a, DatatypeRef):
        if a.sort().name()[:5]=="List_" and isinstance(b,list):
            counts = Counter(b)
            conds=[]
            nums=[]
            for num,bcount in counts.items():
                acount = get_count(a,num)
                conds.append(acount<=bcount)
                nums.append(acount)
            conds.append(Sum(nums)==length(a))#does not contain other number than numbers from b
            return And(conds)

s = Solver()
a=IntList.cons(1, IntList.cons(2, IntList.nil))
s.add(Equal(a,[1,2]))
if s.check() != sat:
    print("IntList.cons(1, IntList.cons(2, IntList.nil)) do not equal to [1,2]")

# print("trying to get 498 from 50,75,25,100,1,10")
# s = Solver()
# tx = Const("tx",Tree)
# s.add(And(is_valid(tx),compute(tx)==499,Equal(get_numbers(tx),[50,75,25,100,1,10])))#takes 1m30s but sometimes take 24min
# if s.check() != sat:
#     print("can not get 498 from 50,75,25,100,1,10")
# m = s.model()
# print("tx = ",m.eval(tx))
# 50 + 75 = 125
# 125 + 25 = 150
# 150 - 100 = 50
# 50 x 10 = 500
# 500 - 1 = 499



s = Solver()
tx = Const("tx",Tree)
#s.add(And(is_valid(tx),compute(tx)==8,is_all_same(tx,7)==2))#take very long time to prove unsatisfiability, at least 852min
#s.add(And(is_valid(tx),compute(tx)!=8,Equal(get_numbers(tx),[7,7])))#TODO also does not ends
s.add(And(is_valid(tx),compute(tx)!=8,Equal(get_numbers(tx),[7,7])))#TODO finds all 12 solutions but does not ends
for x in range(1000):
    st=s.check()
    if st==sat:
        m = s.model()
        print("tx = ",m.eval(tx))
        s.add(tx!=m.eval(tx))
    else:
        print(st)
        break



tx =  node(mul, node(num(7), nil, nil), node(num(-7), nil, nil))
tx =  node(mul, node(num(7), nil, nil), node(num(7), nil, nil))
tx =  node(plus, node(num(7), nil, nil), node(num(-7), nil, nil))
tx =  node(div, node(num(7), nil, nil), node(num(-7), nil, nil))
tx =  node(plus, node(num(-7), nil, nil), node(num(-7), nil, nil))
tx =  node(div, node(num(-7), nil, nil), node(num(-7), nil, nil))
tx =  node(mul, node(num(-7), nil, nil), node(num(-7), nil, nil))
tx =  node(plus, node(num(7), nil, nil), node(num(7), nil, nil))
tx =  node(plus, node(num(-7), nil, nil), node(num(7), nil, nil))
tx =  node(mul, node(num(-7), nil, nil), node(num(7), nil, nil))
tx =  node(div, node(num(-7), nil, nil), node(num(7), nil, nil))
tx =  node(div, node(num(7), nil, nil), node(num(7), nil, nil))


In [28]:
from z3 import *

## Define List
def DeclareList(sort):
    List = Datatype('List_of_%s' % sort.name())
    List.declare('cons', ('car', sort), ('cdr', List))
    List.declare('nil')
    return List.create()

IntList = DeclareList(IntSort())

## Define Rec Function
appendo = RecFunction('appendo', IntList, IntList, IntList)
l = FreshConst(IntList)
s = FreshConst(IntList)
ti = Const("ti",IntSort())
tl = Const("tl",IntList)
RecAddDefinition( appendo
                , [l, s]
                , If(IntList.is_cons(l),
                #, If(Exists([tl,ti], IntList.cons(ti, tl) == l) ,#TODO why do not works when I replace if and else conditions
                 #, If(Or(IntList.car(l) > -1,IntList.car(l) <= -1), #TODO does not work too
                     IntList.cons(IntList.car(l), appendo(IntList.cdr(l), s)),
                     s
                     )
                )

# RecAddDefinition( appendo
#                 , [l, s]
#                 , If(IntList.nil == l ,
#                      s,
#                      IntList.cons(IntList.car(l), appendo(IntList.cdr(l), s))
#                      )
#                 )

a = Const('a', IntList)
b = Const('b', IntList)

solver = Solver()
solver.add(appendo(a, b) == IntList.cons(1, IntList.cons(0, IntList.nil)))

while solver.check() == sat:
    m = solver.model()

    v_a = m.eval(a, model_completion=True)
    v_b = m.eval(b, model_completion=True)

    print("Solution:")
    print("  a = " + str(v_a))
    print("  b = " + str(v_b))

    block = Or(a != v_a, b != v_b)
    solver.add(block)

Solution:
  a = nil
  b = cons(1, cons(0, nil))
Solution:
  a = cons(1, nil)
  b = cons(0, nil)
Solution:
  a = cons(1, cons(0, nil))
  b = nil


In [22]:
from z3 import *

# Custom datatype with two constructors
IntList = Datatype('IntList')
IntList.declare('empty')
IntList.declare('cons', ('head', IntSort()), ('tail', IntList))
IntList = IntList.create()

# Function to check constructor based on constraints
def check_constructor(lst):
    s = Solver()
    s.add(IntList.is_empty(lst))  # Constraint for empty constructor
    if s.check() == sat:
        return "empty"

    s = Solver()
    s.add(IntList.is_cons(lst))  # Constraint for cons constructor
    if s.check() == sat:
        return "cons"

# Example usage
lst1 = IntList.empty
lst2 = IntList.cons(10, IntList.empty)

print(check_constructor(lst1))  # Output: empty
print(check_constructor(lst2))  # Output: cons

empty
cons
